# Feature Stability Analysis

Feature stability measures whether the statistical behavior of a
feature remains reasonably consistent across different data windows.

In this study, feature stability is evaluated using window-level
statistical characteristics such as mean, standard deviation, and
coefficient of variation (CV).

The purpose is to transform the observations obtained during EDA
into quantitative feature reliability indicators.

Important distinction:

- Stability measures consistency of feature behavior.
- Drift measures change in the feature distribution between windows.
- Importance consistency measures whether the model continues to
  rely on the feature consistently.

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [28]:
loan_df = pd.read_csv("..\datasets\loan_data_set.csv")

print("Dataset shape:", loan_df.shape)
loan_df.head()

Dataset shape: (614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [29]:
print(loan_df.shape)
print(loan_df.columns.tolist())
print(loan_df.isnull().sum())

(614, 13)
['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']
Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64


In [25]:
sns.set_theme(style="whitegrid")

In [30]:
target = "Loan_Status"

features = [
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "Loan_Amount_Term",
    "Credit_History"
]

## Create the stability function

In [33]:
def calculate_feature_stability(df, numeric_features, n_windows=5):
    """
    Calculate window-level statistics for feature stability analysis.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.
    numeric_features : list
        Numerical feature names.
    n_windows : int
        Number of sequential windows.

    Returns
    -------
    stability_df : pandas.DataFrame
        Window-level mean, standard deviation and CV.
    """

    data = df[numeric_features].copy()

    # Divide observations into sequential windows
    data["window"] = pd.qcut(
        np.arange(len(data)),
        q=n_windows,
        labels=False,
        duplicates="drop"
    )

    results = []

    for window_id, window_data in data.groupby("window"):

        for feature in numeric_features:

            values = window_data[feature].dropna()

            if len(values) == 0:
                continue

            mean_value = values.mean()
            std_value = values.std()

            # Avoid division by zero
            if mean_value != 0:
                cv_value = std_value / abs(mean_value)
            else:
                cv_value = np.nan

            results.append({
                "Window": window_id + 1,
                "Feature": feature,
                "Mean": mean_value,
                "Std": std_value,
                "CV": cv_value
            })

    return pd.DataFrame(results)

### for the Loan dataset

In [38]:
# Identify numerical features for stability analysis

loan_num_cols = loan_df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical features:")
print(loan_num_cols)

Numerical features:
['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']


In [39]:
loan_stability = calculate_feature_stability(
    loan_df,
    loan_num_cols,
    n_windows=5
)

loan_stability.head(20)

,Window,Feature,Mean,Std,CV
0,1,ApplicantIncome,4418.154472,2611.561583,0.591098
1,1,CoapplicantIncome,1722.666667,1975.246333,1.146621
2,1,LoanAmount,136.643478,61.428491,0.449553
3,1,Loan_Amount_Term,343.076923,60.947427,0.177649
4,1,Credit_History,0.842105,0.366252,0.434924
5,2,ApplicantIncome,6242.747967,7393.081702,1.184267
6,2,CoapplicantIncome,1417.642276,1978.723388,1.395785
7,2,LoanAmount,151.388430,104.842848,0.692542
8,2,Loan_Amount_Term,340.336134,68.592790,0.201544
9,2,Credit_History,0.876106,0.330928,0.377726


In [40]:
loan_stability_pivot = loan_stability.pivot(
    index="Feature",
    columns="Window",
    values="CV"
)

loan_stability_pivot

Window,1,2,3,4,5
Feature,,,,,
ApplicantIncome,0.591098,1.184267,1.192280,1.383571,0.769743
CoapplicantIncome,1.146621,1.395785,1.107363,1.913557,2.372328
Credit_History,0.434924,0.377726,0.410083,0.493879,0.454028
LoanAmount,0.449553,0.692542,0.533524,0.568222,0.613306
Loan_Amount_Term,0.177649,0.201544,0.180578,0.154619,0.231275


### for House Price

In [46]:
# Identify numerical features for stability analysis

house_num_cols = house_df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical features:")
print(house_num_cols)

Numerical features:
['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated']


In [47]:
house_stability = calculate_feature_stability(
    house_df,
    house_num_cols,
    n_windows=5
)

house_stability_pivot = house_stability.pivot(
    index="Feature",
    columns="Window",
    values="CV"
)

house_stability_pivot

Window,1,2,3,4,5
Feature,,,,,
bathrooms,0.363766,0.367485,0.358784,0.358785,0.364108
bedrooms,0.268712,0.273386,0.258715,0.272500,0.262595
condition,0.193134,0.202835,0.193114,0.197892,0.193988
floors,0.366041,0.364821,0.353241,0.349717,0.345775
price,0.604454,0.649253,0.742446,0.651166,1.834986
sqft_above,0.476592,0.475792,0.455301,0.462175,0.483458
sqft_basement,1.485351,1.462926,1.474667,1.471717,1.538536
sqft_living,0.463612,0.444430,0.449689,0.438806,0.452057
sqft_lot,2.309068,3.055533,2.289126,2.294525,1.898525


### for Credit Card

In [49]:
# Identify numerical features for stability analysis

credit_num_cols = credit_df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical features:")
print(credit_num_cols)

Numerical features:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


In [50]:
credit_stability = calculate_feature_stability(
    credit_df,
    credit_num_cols,
    n_windows=5
)

credit_stability_pivot = credit_stability.pivot(
    index="Feature",
    columns="Window",
    values="CV"
)

credit_stability_pivot

Window,1,2,3,4,5
Feature,,,,,
Amount,2.711831,2.705863,3.062911,2.599747,3.162305
Class,18.985411,26.782528,23.115787,31.536389,27.673775
Time,0.435360,0.122135,0.172349,0.054632,0.048361
V1,7.942207,6.886244,308.932835,8.863955,7.228351
V10,22.142583,120.291626,22.135502,13.000848,275.392237
V11,3.230634,8.293722,8.717880,3.321919,2.961584
V12,4.661448,3.506003,5.556395,6.396238,3.900956
V13,8.674039,6.333887,5.015513,7.125801,13.487370
V14,7.897317,15.301097,6.261373,7.766946,6.901834


In [51]:
def calculate_stability_index(stability_df):

    summary = (
        stability_df
        .groupby("Feature")["CV"]
        .agg(["mean", "std"])
        .reset_index()
    )

    summary["CV_Variation"] = (
        summary["std"] /
        summary["mean"].abs().replace(0, np.nan)
    )

    summary["Stability_Index"] = (
        1 / (1 + summary["CV_Variation"])
    )

    return summary.sort_values(
        "Stability_Index",
        ascending=True
    )

In [52]:
loan_stability_index = calculate_stability_index(
    loan_stability
)

loan_stability_index

,Feature,mean,std,CV_Variation,Stability_Index
1,CoapplicantIncome,1.587131,0.544042,0.342783,0.744722
0,ApplicantIncome,1.024192,0.329904,0.322111,0.756366
3,LoanAmount,0.571429,0.090447,0.158283,0.863347
4,Loan_Amount_Term,0.189133,0.028837,0.152469,0.867702
2,Credit_History,0.434128,0.043950,0.101236,0.908070


# SECTION 10.1 — Feature Stability Assessment: Methodological Update

## Purpose

The initial Feature Stability analysis was performed by dividing the dataset
into sequential windows and calculating the coefficient of variation (CV)
for each numerical feature within each window.

The initial results demonstrate that feature behavior can vary considerably
across data windows. However, the results also show that CV alone is not
sufficient to represent feature stability for all types of numerical
features.

In particular, CV can become extremely large when the feature mean is close
to zero. This behavior was observed for several anonymized features in the
Credit Card Fraud dataset. Therefore, CV will be treated as one component
of the stability assessment rather than as the sole stability metric.

---

## Updated Multi-Statistic Feature Stability Assessment

To obtain a more robust measurement of feature stability, the framework
will evaluate multiple statistical characteristics of each feature across
data windows.

### A. Mean Stability

The mean of each numerical feature will be calculated independently for
each data window.

The variation in window-level means will be used to determine whether the
central tendency of a feature remains consistent across windows.

A feature with relatively small changes in its mean across windows will
provide stronger evidence of stability.

---

### B. Standard-Deviation Stability

The standard deviation of each numerical feature will be calculated for
each data window.

Changes in standard deviation will be examined to determine whether the
spread or variability of the feature remains consistent across windows.

A feature whose standard deviation changes substantially between windows
may indicate unstable statistical behavior.

---

### C. Coefficient-of-Variation (CV) Stability

The coefficient of variation will be calculated as:

    CV = Standard Deviation / |Mean|

CV provides a scale-independent measure of relative variability.

However, CV will only be interpreted when the feature mean is sufficiently
different from zero. Features with means close to zero can produce
artificially large CV values.

Therefore, CV will be used as a supporting stability indicator rather than
as the sole basis for determining feature reliability.

---

### D. Distributional Stability

In addition to summary statistics, the framework will later evaluate
whether the overall feature distribution remains consistent across
different data windows.

Distributional comparison will be implemented using appropriate
statistical/distribution-based measures during the subsequent Feature
Drift analysis.

This distinction is important:

- Statistical stability evaluates changes in summary characteristics.
- Distributional drift evaluates changes in the overall feature
  distribution.
- Feature importance consistency evaluates whether the model continues
  to rely on the feature consistently.

---

## Current Interpretation

The initial stability experiment should therefore be considered an
exploratory quantitative assessment rather than the final Feature
Reliability Score.

The current results indicate that:

- Some features exhibit relatively consistent statistical behavior across
  windows.
- Some features show substantial variation between windows.
- CV can be informative for certain features but can become unstable when
  feature means approach zero.
- Different types of numerical features may require different stability
  considerations.

Therefore, the framework will use a multi-statistic approach rather than
relying on a single metric.

---

## Research Direction

The revised Feature Stability component will follow:

    Feature
       |
       +---- Mean Stability
       |
       +---- Standard-Deviation Stability
       |
       +---- CV Variation
       |
       +---- Distributional Stability
                    |
                    v
          Feature Stability Assessment

The final Feature Reliability Score (FRS) will NOT be defined at this
stage.

Instead, Feature Stability will first be evaluated independently and will
later be combined with other reliability dimensions such as:

- Feature Drift
- Feature Importance Consistency
- Correlation Consistency

This ensures that a feature is not classified as unreliable based on a
single statistical characteristic.

---

## Important Methodological Principle

A low value on one stability indicator does not automatically imply that
a feature is unreliable.

Feature reliability will be determined using multiple complementary
evidence sources and will ultimately be evaluated against downstream
machine-learning model behavior and performance.

Therefore, the framework follows:

    Data Quality
          |
          v
    Feature Behavior
          |
          v
    Feature Reliability
          |
          v
    Model Behavior
          |
          v
    Model Performance

### Create the revised stability function

In [53]:
def calculate_multistatistic_stability(
    df,
    numeric_features,
    n_windows=5,
    mean_threshold=1e-6
):
    """
    Calculate multi-statistic feature stability across sequential windows.

    Metrics:
    - Window Mean
    - Window Standard Deviation
    - Coefficient of Variation (when mean is sufficiently far from zero)
    """

    data = df[numeric_features].copy()

    # Create sequential windows
    data["window"] = pd.qcut(
        np.arange(len(data)),
        q=n_windows,
        labels=False,
        duplicates="drop"
    )

    results = []

    for window_id, window_data in data.groupby("window"):

        for feature in numeric_features:

            values = window_data[feature].dropna()

            if len(values) == 0:
                continue

            mean_value = values.mean()
            std_value = values.std()

            # CV is only meaningful when mean is sufficiently
            # far from zero
            if abs(mean_value) > mean_threshold:
                cv_value = std_value / abs(mean_value)
            else:
                cv_value = np.nan

            results.append({
                "Window": window_id + 1,
                "Feature": feature,
                "Mean": mean_value,
                "Std": std_value,
                "CV": cv_value
            })

    return pd.DataFrame(results)

### Run it on Loan

In [54]:
loan_stability_multi = calculate_multistatistic_stability(
    loan_df,
    loan_num_cols,
    n_windows=5
)

loan_stability_multi.head(20)

,Window,Feature,Mean,Std,CV
0,1,ApplicantIncome,4418.154472,2611.561583,0.591098
1,1,CoapplicantIncome,1722.666667,1975.246333,1.146621
2,1,LoanAmount,136.643478,61.428491,0.449553
3,1,Loan_Amount_Term,343.076923,60.947427,0.177649
4,1,Credit_History,0.842105,0.366252,0.434924
5,2,ApplicantIncome,6242.747967,7393.081702,1.184267
6,2,CoapplicantIncome,1417.642276,1978.723388,1.395785
7,2,LoanAmount,151.388430,104.842848,0.692542
8,2,Loan_Amount_Term,340.336134,68.592790,0.201544
9,2,Credit_History,0.876106,0.330928,0.377726


### Mean across windows

In [55]:
loan_mean_windows = loan_stability_multi.pivot(
    index="Feature",
    columns="Window",
    values="Mean"
)

loan_mean_windows

Window,1,2,3,4,5
Feature,,,,,
ApplicantIncome,4418.154472,6242.747967,5300.254098,6010.463415,5044.837398
CoapplicantIncome,1722.666667,1417.642276,1384.367213,1502.675772,2076.951220
Credit_History,0.842105,0.876106,0.857143,0.805310,0.830357
LoanAmount,136.643478,151.388430,147.016949,140.932773,155.672269
Loan_Amount_Term,343.076923,340.336134,348.500000,342.148760,336.097561


### Standard deviation across windows

In [56]:
loan_std_windows = loan_stability_multi.pivot(
    index="Feature",
    columns="Window",
    values="Std"
)

loan_std_windows

Window,1,2,3,4,5
Feature,,,,,
ApplicantIncome,2611.561583,7393.081702,6319.386917,8315.902442,3883.226933
CoapplicantIncome,1975.246333,1978.723388,1532.996509,2875.455910,4927.208662
Credit_History,0.366252,0.330928,0.351500,0.397726,0.377006
LoanAmount,61.428491,104.842848,78.437082,80.081075,95.474738
Loan_Amount_Term,60.947427,68.592790,62.931335,52.902530,77.730874


### CV across windows

In [57]:
loan_cv_windows = loan_stability_multi.pivot(
    index="Feature",
    columns="Window",
    values="CV"
)

loan_cv_windows

Window,1,2,3,4,5
Feature,,,,,
ApplicantIncome,0.591098,1.184267,1.192280,1.383571,0.769743
CoapplicantIncome,1.146621,1.395785,1.107363,1.913557,2.372328
Credit_History,0.434924,0.377726,0.410083,0.493879,0.454028
LoanAmount,0.449553,0.692542,0.533524,0.568222,0.613306
Loan_Amount_Term,0.177649,0.201544,0.180578,0.154619,0.231275


## Calculate relative mean variation

In [58]:
def calculate_stability_summary(stability_df):

    summary = (
        stability_df
        .groupby("Feature")
        .agg(
            Mean_of_Means=("Mean", "mean"),
            Std_of_Means=("Mean", "std"),
            Mean_of_Std=("Std", "mean"),
            Std_of_Std=("Std", "std"),
            Mean_CV=("CV", "mean"),
            Std_CV=("CV", "std")
        )
        .reset_index()
    )

    # Relative variation in window means
    summary["Mean_Variation"] = (
        summary["Std_of_Means"] /
        summary["Mean_of_Means"].abs().replace(0, np.nan)
    )

    # Relative variation in window standard deviations
    summary["Std_Variation"] = (
        summary["Std_of_Std"] /
        summary["Mean_of_Std"].abs().replace(0, np.nan)
    )

    # Relative variation in CV
    summary["CV_Variation"] = (
        summary["Std_CV"] /
        summary["Mean_CV"].abs().replace(0, np.nan)
    )

    return summary

In [59]:
loan_stability_summary = calculate_stability_summary(
    loan_stability_multi
)

loan_stability_summary

,Feature,Mean_of_Means,Std_of_Means,Mean_of_Std,Std_of_Std,Mean_CV,Std_CV,Mean_Variation,Std_Variation,CV_Variation
0,ApplicantIncome,5403.291470,738.739790,5704.631916,2394.374676,1.024192,0.329904,0.136720,0.419725,0.322111
1,CoapplicantIncome,1620.860630,287.044244,2657.926160,1359.209066,1.587131,0.544042,0.177094,0.511380,0.342783
2,Credit_History,0.842204,0.026805,0.364682,0.025295,0.434128,0.043950,0.031827,0.069362,0.101236
3,LoanAmount,146.330780,7.685967,84.052847,16.747067,0.571429,0.090447,0.052525,0.199244,0.158283
4,Loan_Amount_Term,342.031876,4.500946,64.620991,9.237377,0.189133,0.028837,0.013159,0.142947,0.152469


### Run the same analysis for House Price

In [60]:
house_stability_multi = calculate_multistatistic_stability(
    house_df,
    house_num_cols,
    n_windows=5
)

house_stability_summary = calculate_stability_summary(
    house_stability_multi
)

house_stability_summary

,Feature,Mean_of_Means,Std_of_Means,Mean_of_Std,Std_of_Std,Mean_CV,Std_CV,Mean_Variation,Std_Variation,CV_Variation
0,bathrooms,2.160815,0.033282,0.783464,0.013446,0.362585,0.003762,0.015402,0.017162,0.010376
1,bedrooms,3.400870,0.039239,0.908480,0.013919,0.267182,0.006361,0.011538,0.015321,0.023807
2,condition,3.451739,0.014696,0.677236,0.016499,0.196192,0.004204,0.004258,0.024362,0.021430
3,floors,1.512065,0.023982,0.538037,0.008831,0.355919,0.009086,0.015861,0.016413,0.025528
4,price,551962.988473,8612.620716,496951.835219,298256.984040,0.896461,0.527043,0.015604,0.600173,0.587916
5,sqft_above,1827.265435,58.047779,860.216576,40.486103,0.470664,0.011546,0.031768,0.047065,0.024531
6,sqft_basement,312.081522,14.472562,463.821538,19.771481,1.486639,0.030097,0.046374,0.042627,0.020245
7,sqft_living,2139.346957,51.819863,962.112564,30.903756,0.449719,0.009294,0.024222,0.032121,0.020667
8,sqft_lot,14852.516087,844.231862,35301.271291,7251.133663,2.369355,0.420771,0.056841,0.205407,0.177589
9,view,0.240652,0.030576,0.776557,0.057578,3.246877,0.203204,0.127055,0.074145,0.062585


### Run it for Credit Card

In [61]:
credit_stability_multi = calculate_multistatistic_stability(
    credit_df,
    credit_num_cols,
    n_windows=5
)

credit_stability_summary = calculate_stability_summary(
    credit_stability_multi
)

credit_stability_summary

,Feature,Mean_of_Means,Std_of_Means,Mean_of_Std,Std_of_Std,Mean_CV,Std_CV,Mean_Variation,Std_Variation,CV_Variation
0,Amount,88.472664,12.551653,249.668180,17.338933,2.848532,0.247663,0.141870,0.069448,0.086944
1,Class,0.001667,0.000688,0.040152,0.008044,25.618778,4.766077,0.412971,0.200346,0.186038
2,Time,94811.302453,51659.118945,10319.197343,4034.156869,0.166567,0.158713,0.544862,0.390937,0.952845
3,V1,0.005918,0.254175,1.930110,0.149370,67.970719,134.704047,42.949516,0.077389,1.981795
4,V10,-0.001441,0.055215,1.073449,0.070145,90.592559,112.277777,38.329585,0.065345,1.239371
5,V11,0.000201,0.279530,0.984666,0.084610,5.305148,2.928627,1393.807105,0.085928,0.552035
6,V12,-0.000714,0.231291,0.933945,0.304832,4.804208,1.185530,324.020896,0.326392,0.246769
7,V13,0.000603,0.161482,0.980691,0.101722,8.127322,3.276119,267.828395,0.103725,0.403099
8,V14,0.000252,0.140599,0.937228,0.125072,8.825714,3.680715,558.304325,0.133449,0.417044
9,V15,0.001042,0.204463,0.894168,0.071239,48.183460,97.366951,196.154885,0.079671,2.020755


### Add a diagnostic for near-zero means

In [62]:
def identify_near_zero_mean_features(
    stability_df,
    threshold=1e-3
):
    diagnostic = (
        stability_df
        .groupby("Feature")["Mean"]
        .agg(
            Mean_Min="min",
            Mean_Max="max",
            Mean_Average="mean"
        )
        .reset_index()
    )

    diagnostic["Near_Zero_Mean"] = (
        diagnostic["Mean_Average"].abs() < threshold
    )

    return diagnostic.sort_values(
        "Mean_Average",
        key=lambda x: x.abs()
    )

In [63]:
credit_mean_diagnostic = identify_near_zero_mean_features(
    credit_stability_multi
)

credit_mean_diagnostic

,Feature,Mean_Min,Mean_Max,Mean_Average,Near_Zero_Mean
17,V22,-0.114657,0.159000,-0.000015,True
21,V26,-0.027172,0.033447,0.000149,True
11,V17,-0.065568,0.100848,0.000170,True
15,V20,-0.052727,0.048317,0.000187,True
18,V23,-0.039097,0.040363,0.000198,True
5,V11,-0.305148,0.342475,0.000201,True
19,V24,-0.022077,0.015075,0.000214,True
20,V25,-0.152739,0.135460,-0.000233,True
8,V14,-0.132497,0.162537,0.000252,True
13,V19,-0.026516,0.029728,-0.000264,True


# SECTION 10.2 — Feature-Type-Aware Stability Assessment

## Motivation

The initial multi-statistic stability analysis evaluated feature behavior
using:

- Window-level mean
- Window-level standard deviation
- Coefficient of variation (CV)

The analysis demonstrated that these statistics provide useful information
about feature behavior across data windows. However, it also revealed an
important limitation:

> A single stability statistic cannot be interpreted uniformly across all
> numerical features and datasets.

This is particularly evident for near-zero-centered features and rare
binary features, where the coefficient of variation can become extremely
large even when the underlying feature behavior may not indicate equivalent
levels of instability.

Therefore, the Feature Reliability Framework will adopt a
**feature-type-aware stability assessment**.

---

## 1. Continuous Positive-Valued Features

For continuous features whose mean is sufficiently far from zero, the
following statistics will be considered:

- Mean variation
- Standard-deviation variation
- CV variation

The coefficient of variation is useful in this case because it measures
relative variability independently of the feature's measurement scale.

The CV is defined as:

    CV = σ / |μ|

where:

- σ = standard deviation
- μ = mean

However, CV will only be interpreted when the mean is sufficiently far
from zero.

---

## 2. Near-Zero-Centered Continuous Features

For features whose mean is close to zero, CV becomes numerically unstable
because the denominator approaches zero.

This behavior was observed in the Credit Card Fraud dataset for several
anonymized V-features.

For such features:

- Mean variation will be evaluated.
- Standard-deviation variation will be evaluated.
- CV will be flagged as unreliable or excluded from the primary stability
  interpretation.

Therefore:

    Near-zero mean
          ↓
    CV becomes unstable
          ↓
    Do not interpret large CV as direct evidence of feature instability
          ↓
    Use mean and standard-deviation behavior instead

This prevents mathematical artifacts from being incorrectly interpreted
as feature unreliability.

---

## 3. Binary Features

Binary features require a different interpretation because their values
represent discrete states rather than continuous measurements.

For a binary feature, the primary stability characteristic is the
proportion of observations belonging to each state.

Therefore, binary feature stability will be evaluated through:

- Window-level prevalence/proportion
- Variation in prevalence across windows
- Distributional consistency

The coefficient of variation will not be used as the primary stability
measure for binary features.

---

## 4. Ordinal and Discrete Features

Features with a limited number of discrete or ordinal values will be
treated separately from continuous variables.

Examples may include:

- Number of bedrooms
- Condition rating
- Floors
- Credit history indicators

For these variables, the framework will emphasize:

- Window-level mean
- Distribution/proportion of categories
- Distributional stability

rather than relying exclusively on CV.

---

## 5. Target Variables

Target variables will not be treated as ordinary predictive features.

For example:

- `Class` in the Credit Card Fraud dataset represents the target.
- It should therefore not contribute directly to the predictor Feature
  Stability Score.

However, target distribution changes may later be examined separately as
part of dataset-level drift or data-quality analysis.

---

## 6. Temporal Variables

Temporal variables require special consideration.

For example:

- `Time` in the Credit Card Fraud dataset represents temporal ordering.

It should not be interpreted in exactly the same way as an ordinary
predictive numerical feature.

Temporal variables will therefore be retained for:

- Window construction
- Temporal ordering
- Drift analysis

but will not automatically receive the same Feature Stability
interpretation as ordinary continuous predictors.

---

# Feature-Type-Aware Stability Decision Logic

The revised framework will follow:

    Feature
       |
       v
    Identify Feature Type
       |
       +-------------------+
       |                   |
       v                   v
   Continuous          Discrete/Binary
       |                   |
       v                   v
   Check Mean          Distribution /
   near zero?          Proportion
       |
   +---+---+
   |       |
  No      Yes
   |       |
   v       v
Mean + SD + CV       Mean + SD
   |                   |
   +---------+---------+
             |
             v
    Distributional Stability
             |
             v
     Stability Profile

---

# Research Interpretation

The purpose of this feature-type-aware approach is not to artificially
force every feature into a single statistical formula.

Instead, the framework attempts to measure feature reliability using
statistics that are appropriate for the characteristics of the feature.

This supports the central data-centric principle of the research:

> **Feature reliability should be evaluated based on feature behavior,
> data characteristics, and behavior across data windows rather than by
> applying a single model-independent statistic to every feature.**

---

# Current Research Finding

The experiments conducted so far provide preliminary evidence that:

1. Feature behavior varies across data windows.
2. Mean and standard-deviation variation provide complementary information.
3. CV is useful for many continuous features but is sensitive to
   near-zero means.
4. Rare binary features can also produce misleading CV values.
5. Therefore, feature stability requires a multi-statistic assessment.
6. The interpretation of these statistics should depend on feature type.

These findings motivate the next stage of the framework:

**Feature-Type-Aware Stability Measurement.**

---

# Position in the Overall Framework

The current research pipeline is now:

    Raw Data
        ↓
    Data Quality Assessment
        ↓
    Feature Engineering
        ↓
    Feature-Type Identification
        ↓
    Window-Based Feature Analysis
        ↓
    Multi-Statistic Stability Assessment
        ↓
    Distributional Stability
        ↓
    Feature Stability Profile
        ↓
    Feature Drift Analysis
        ↓
    Feature Importance Consistency
        ↓
    Correlation Consistency
        ↓
    Feature Reliability Score (FRS)
        ↓
    ML Performance Evaluation

The Feature Reliability Score will be defined only after the individual
reliability dimensions have been independently evaluated and validated.

# SECTION 10.3 — Implement Feature-Type-Aware Stability

### Define feature types

In [64]:
def identify_feature_type(
    series,
    near_zero_threshold=0.01,
    discrete_unique_threshold=10
):
    """
    Identify a numerical feature's statistical type.

    Categories:
    - binary
    - discrete
    - near_zero_continuous
    - continuous
    """

    s = series.dropna()

    if len(s) == 0:
        return "unknown"

    unique_values = s.nunique()

    # Binary feature
    if unique_values <= 2:
        return "binary"

    # Integer-like low-cardinality feature
    if (
        pd.api.types.is_integer_dtype(s)
        and unique_values <= discrete_unique_threshold
    ):
        return "discrete"

    # Near-zero centered continuous feature
    if abs(s.mean()) < near_zero_threshold:
        return "near_zero_continuous"

    return "continuous"

### Create a feature-type profile

In [65]:
def create_feature_type_profile(
    df,
    numeric_features
):
    profile = []

    for feature in numeric_features:

        feature_type = identify_feature_type(
            df[feature]
        )

        profile.append({
            "Feature": feature,
            "Data_Type": str(df[feature].dtype),
            "Unique_Values": df[feature].nunique(),
            "Mean": df[feature].mean(),
            "Feature_Type": feature_type
        })

    return pd.DataFrame(profile)

### Loan feature profile

In [66]:
loan_feature_profile = create_feature_type_profile(
    loan_df,
    loan_num_cols
)

loan_feature_profile

,Feature,Data_Type,Unique_Values,Mean,Feature_Type
0,ApplicantIncome,int64,505,5403.459283,continuous
1,CoapplicantIncome,float64,287,1621.245798,continuous
2,LoanAmount,float64,203,146.412162,continuous
3,Loan_Amount_Term,float64,10,342.000000,continuous
4,Credit_History,float64,2,0.842199,binary


### House Price feature profile

In [67]:
house_feature_profile = create_feature_type_profile(
    house_df,
    house_num_cols
)

house_feature_profile

,Feature,Data_Type,Unique_Values,Mean,Feature_Type
0,price,float64,1741,551962.988473,continuous
1,bedrooms,float64,10,3.400870,continuous
2,bathrooms,float64,26,2.160815,continuous
3,sqft_living,int64,566,2139.346957,continuous
4,sqft_lot,int64,3113,14852.516087,continuous
5,floors,float64,6,1.512065,continuous
6,waterfront,int64,2,0.007174,binary
7,view,int64,5,0.240652,discrete
8,condition,int64,5,3.451739,discrete
9,sqft_above,int64,511,1827.265435,continuous


### Credit Card feature profile

In [68]:
credit_feature_profile = create_feature_type_profile(
    credit_df,
    credit_num_cols
)

credit_feature_profile

,Feature,Data_Type,Unique_Values,Mean,Feature_Type
0,Time,float64,124592,94811.077600,continuous
1,V1,float64,275663,0.005917,near_zero_continuous
2,V2,float64,275663,-0.004135,near_zero_continuous
3,V3,float64,275663,0.001613,near_zero_continuous
4,V4,float64,275663,-0.002966,near_zero_continuous
5,V5,float64,275663,0.001828,near_zero_continuous
6,V6,float64,275663,-0.001139,near_zero_continuous
7,V7,float64,275663,0.001801,near_zero_continuous
8,V8,float64,275663,-0.000854,near_zero_continuous
9,V9,float64,275663,-0.001596,near_zero_continuous


### Define research roles

In [69]:
dataset_roles = {
    "Loan": {
        "target": "Loan_Status",
        "temporal": []
    },

    "House": {
        "target": "price",
        "temporal": []
    },

    "Credit": {
        "target": "Class",
        "temporal": ["Time"]
    }
}

### Build the stability assessment

In [70]:
def feature_type_aware_stability(
    df,
    numeric_features,
    n_windows=5,
    target=None,
    temporal_features=None,
    near_zero_threshold=0.01
):
    """
    Feature-type-aware stability assessment.

    Measures:
    - Mean variation
    - Standard-deviation variation
    - CV variation where appropriate
    - Feature type

    Targets and explicitly defined temporal variables are excluded
    from the predictor stability assessment.
    """

    if temporal_features is None:
        temporal_features = []

    # Remove target and temporal variables
    analysis_features = [
        f for f in numeric_features
        if f != target and f not in temporal_features
    ]

    results = []

    # Sequential windows
    data = df[analysis_features].copy()

    data["window"] = pd.qcut(
        np.arange(len(data)),
        q=n_windows,
        labels=False,
        duplicates="drop"
    )

    for feature in analysis_features:

        feature_type = identify_feature_type(
            df[feature],
            near_zero_threshold=near_zero_threshold
        )

        for window_id, window_data in data.groupby("window"):

            values = window_data[feature].dropna()

            if len(values) == 0:
                continue

            mean_value = values.mean()
            std_value = values.std()

            # CV only for appropriate continuous features
            if (
                feature_type == "continuous"
                and abs(mean_value) >= near_zero_threshold
            ):
                cv_value = (
                    std_value / abs(mean_value)
                )
            else:
                cv_value = np.nan

            results.append({
                "Window": window_id + 1,
                "Feature": feature,
                "Feature_Type": feature_type,
                "Mean": mean_value,
                "Std": std_value,
                "CV": cv_value
            })

    return pd.DataFrame(results)

### Run it for Loan

In [71]:
loan_stability_v2 = feature_type_aware_stability(
    df=loan_df,
    numeric_features=loan_num_cols,
    n_windows=5,
    target="Loan_Status"
)

loan_stability_v2.head(20)

,Window,Feature,Feature_Type,Mean,Std,CV
0,1,ApplicantIncome,continuous,4418.154472,2611.561583,0.591098
1,2,ApplicantIncome,continuous,6242.747967,7393.081702,1.184267
2,3,ApplicantIncome,continuous,5300.254098,6319.386917,1.192280
3,4,ApplicantIncome,continuous,6010.463415,8315.902442,1.383571
4,5,ApplicantIncome,continuous,5044.837398,3883.226933,0.769743
5,1,CoapplicantIncome,continuous,1722.666667,1975.246333,1.146621
6,2,CoapplicantIncome,continuous,1417.642276,1978.723388,1.395785
7,3,CoapplicantIncome,continuous,1384.367213,1532.996509,1.107363
8,4,CoapplicantIncome,continuous,1502.675772,2875.455910,1.913557
9,5,CoapplicantIncome,continuous,2076.951220,4927.208662,2.372328


In [72]:
loan_stability_v2_summary = calculate_stability_summary(
    loan_stability_v2
)

loan_stability_v2_summary

,Feature,Mean_of_Means,Std_of_Means,Mean_of_Std,Std_of_Std,Mean_CV,Std_CV,Mean_Variation,Std_Variation,CV_Variation
0,ApplicantIncome,5403.291470,738.739790,5704.631916,2394.374676,1.024192,0.329904,0.136720,0.419725,0.322111
1,CoapplicantIncome,1620.860630,287.044244,2657.926160,1359.209066,1.587131,0.544042,0.177094,0.511380,0.342783
2,Credit_History,0.842204,0.026805,0.364682,0.025295,NaN,NaN,0.031827,0.069362,NaN
3,LoanAmount,146.330780,7.685967,84.052847,16.747067,0.571429,0.090447,0.052525,0.199244,0.158283
4,Loan_Amount_Term,342.031876,4.500946,64.620991,9.237377,0.189133,0.028837,0.013159,0.142947,0.152469


### Run it for House Price

In [73]:
house_stability_v2 = feature_type_aware_stability(
    df=house_df,
    numeric_features=house_num_cols,
    n_windows=5,
    target="price"
)

house_stability_v2.head(20)

,Window,Feature,Feature_Type,Mean,Std,CV
0,1,bedrooms,continuous,3.398913,0.913327,0.268712
1,2,bedrooms,continuous,3.341304,0.913465,0.273386
2,3,bedrooms,continuous,3.429348,0.887225,0.258715
3,4,bedrooms,continuous,3.392391,0.924428,0.272500
4,5,bedrooms,continuous,3.442391,0.903955,0.262595
5,1,bathrooms,continuous,2.132609,0.775770,0.363766
6,2,bathrooms,continuous,2.136957,0.785300,0.367485
7,3,bathrooms,continuous,2.177989,0.781427,0.358784
8,4,bathrooms,continuous,2.145380,0.769730,0.358785
9,5,bathrooms,continuous,2.211141,0.805093,0.364108


In [74]:
house_stability_v2_summary = calculate_stability_summary(
    house_stability_v2
)

house_stability_v2_summary

,Feature,Mean_of_Means,Std_of_Means,Mean_of_Std,Std_of_Std,Mean_CV,Std_CV,Mean_Variation,Std_Variation,CV_Variation
0,bathrooms,2.160815,0.033282,0.783464,0.013446,0.362585,0.003762,0.015402,0.017162,0.010376
1,bedrooms,3.400870,0.039239,0.908480,0.013919,0.267182,0.006361,0.011538,0.015321,0.023807
2,condition,3.451739,0.014696,0.677236,0.016499,NaN,NaN,0.004258,0.024362,NaN
3,floors,1.512065,0.023982,0.538037,0.008831,0.355919,0.009086,0.015861,0.016413,0.025528
4,sqft_above,1827.265435,58.047779,860.216576,40.486103,0.470664,0.011546,0.031768,0.047065,0.024531
5,sqft_basement,312.081522,14.472562,463.821538,19.771481,1.486639,0.030097,0.046374,0.042627,0.020245
6,sqft_living,2139.346957,51.819863,962.112564,30.903756,0.449719,0.009294,0.024222,0.032121,0.020667
7,sqft_lot,14852.516087,844.231862,35301.271291,7251.133663,2.369355,0.420771,0.056841,0.205407,0.177589
8,view,0.240652,0.030576,0.776557,0.057578,NaN,NaN,0.127055,0.074145,NaN
9,waterfront,0.007174,0.001819,0.083820,0.011279,NaN,NaN,0.253533,0.134561,NaN


### Run it for Credit Card

In [75]:
credit_stability_v2 = feature_type_aware_stability(
    df=credit_df,
    numeric_features=credit_num_cols,
    n_windows=5,
    target="Class",
    temporal_features=["Time"]
)

credit_stability_v2.head(20)

,Window,Feature,Feature_Type,Mean,Std,CV
0,1,V1,near_zero_continuous,-0.234379,1.861490,NaN
1,2,V1,near_zero_continuous,-0.261322,1.799529,NaN
2,3,V1,near_zero_continuous,0.005943,1.835987,NaN
3,4,V1,near_zero_continuous,0.244260,2.165108,NaN
4,5,V1,near_zero_continuous,0.275089,1.988438,NaN
5,1,V2,near_zero_continuous,-0.012609,1.640569,NaN
6,2,V2,near_zero_continuous,-0.028905,1.622868,NaN
7,3,V2,near_zero_continuous,0.150549,1.541582,NaN
8,4,V2,near_zero_continuous,-0.180091,1.780907,NaN
9,5,V2,near_zero_continuous,0.050382,1.620656,NaN


In [76]:
credit_stability_v2_summary = calculate_stability_summary(
    credit_stability_v2
)

credit_stability_v2_summary

,Feature,Mean_of_Means,Std_of_Means,Mean_of_Std,Std_of_Std,Mean_CV,Std_CV,Mean_Variation,Std_Variation,CV_Variation
0,Amount,88.472664,12.551653,249.668180,17.338933,2.848532,0.247663,0.141870,0.069448,0.086944
1,V1,0.005918,0.254175,1.930110,0.149370,NaN,NaN,42.949516,0.077389,NaN
2,V10,-0.001441,0.055215,1.073449,0.070145,NaN,NaN,38.329585,0.065345,NaN
3,V11,0.000201,0.279530,0.984666,0.084610,NaN,NaN,1393.807105,0.085928,NaN
4,V12,-0.000714,0.231291,0.933945,0.304832,NaN,NaN,324.020896,0.326392,NaN
5,V13,0.000603,0.161482,0.980691,0.101722,NaN,NaN,267.828395,0.103725,NaN
6,V14,0.000252,0.140599,0.937228,0.125072,NaN,NaN,558.304325,0.133449,NaN
7,V15,0.001042,0.204463,0.894168,0.071239,NaN,NaN,196.154885,0.079671,NaN
8,V16,0.001162,0.015063,0.872720,0.043791,NaN,NaN,12.962697,0.050177,NaN
9,V17,0.000170,0.068334,0.829708,0.148646,NaN,NaN,402.421386,0.179155,NaN


### calculate_stability_summary()

In [77]:
def calculate_stability_summary_v2(stability_df):

    summary = (
        stability_df
        .groupby(["Feature", "Feature_Type"])
        .agg(
            Mean_of_Means=("Mean", "mean"),
            Std_of_Means=("Mean", "std"),
            Mean_of_Std=("Std", "mean"),
            Std_of_Std=("Std", "std"),
            Mean_CV=("CV", "mean"),
            Std_CV=("CV", "std")
        )
        .reset_index()
    )

    # Relative variation in window means
    summary["Mean_Variation"] = (
        summary["Std_of_Means"] /
        summary["Mean_of_Means"].abs().replace(0, np.nan)
    )

    # Relative variation in window standard deviations
    summary["Std_Variation"] = (
        summary["Std_of_Std"] /
        summary["Mean_of_Std"].abs().replace(0, np.nan)
    )

    # Relative CV variation
    summary["CV_Variation"] = (
        summary["Std_CV"] /
        summary["Mean_CV"].abs().replace(0, np.nan)
    )

    return summary

In [78]:
loan_stability_v2_summary = calculate_stability_summary_v2(
    loan_stability_v2
)

house_stability_v2_summary = calculate_stability_summary_v2(
    house_stability_v2
)

credit_stability_v2_summary = calculate_stability_summary_v2(
    credit_stability_v2
)

# Feature Stability Assessment — Multi-Statistic Framework

## Research Objective

The objective of this stage is to assess the **temporal/statistical stability of features** across multiple data windows.

Rather than defining feature stability using a single statistic, this study adopts a **multi-statistic stability assessment**. This is necessary because a feature may maintain a similar mean while its variability changes substantially, or may exhibit stable variance while its overall distribution changes.

The analysis therefore evaluates feature behavior using:

1. Mean stability
2. Standard-deviation stability
3. Coefficient-of-Variation (CV) stability
4. Distributional stability — to be introduced in the next stage

---

## 1. Mean Stability

For each feature, the mean is calculated independently within every observation window.

For feature $f$ and window $w$:

$$
\mu_{f,w} = \frac{1}{n_w}\sum_{i=1}^{n_w}x_{i,f}
$$

The following statistics are then calculated across windows:

- **Mean of Means**
- **Standard Deviation of Means**
- **Mean Variation**

The standard deviation of window means captures how much the feature's central tendency changes across windows.

A lower variation indicates greater stability in the feature's central tendency.

---

## 2. Standard-Deviation Stability

Feature variability is assessed independently within each window.

For feature $f$ and window $w$:

$$
\sigma_{f,w}
=
\sqrt{
\frac{1}{n_w-1}
\sum_{i=1}^{n_w}
(x_{i,f}-\mu_{f,w})^2
}
$$

Across windows, we calculate:

- **Mean of Standard Deviations**
- **Standard Deviation of Standard Deviations**
- **Standard-Deviation Variation**

This provides a second dimension of stability.

A feature may have a stable mean but an unstable standard deviation. Therefore, mean stability alone is insufficient for determining feature reliability.

---

## 3. Coefficient of Variation Stability

For numerical features where the mean is sufficiently far from zero, the coefficient of variation is calculated as:

$$
CV_{f,w} = \frac{\sigma_{f,w}}{|\mu_{f,w}|}
$$

The absolute value of the mean is used in the denominator to avoid sign-related interpretation problems.

Across windows, the following are calculated:

- **Mean CV**
- **Standard Deviation of CV**
- **CV Variation**

CV provides a scale-normalized measure of variability and allows features with different units and magnitudes to be compared more meaningfully.

---

## 4. Near-Zero Mean Handling

CV becomes unstable or mathematically uninformative when the feature mean is close to zero.

This is particularly important for the **Credit Card Fraud dataset**, where many anonymized variables (`V1`–`V28`) have means close to zero.

Therefore:

> CV is **not used as a reliability indicator for near-zero-mean features**.

These features are retained for the other stability measurements rather than being discarded.

This prevents artificially large or undefined CV values from incorrectly indicating feature instability.

The near-zero-mean diagnostic is explicitly recorded using:

- Minimum window mean
- Maximum window mean
- Average window mean
- Near-zero-mean flag

This distinction is important because a large CV for a near-zero-mean feature does **not necessarily represent meaningful instability**.

---

## 5. Current Multi-Statistic Stability Profile

The current analysis therefore produces the following evidence for each feature:

| Stability Dimension | Statistics |
|---|---|
| **Mean Stability** | Mean of Means, Std of Means, Mean Variation |
| **Variance Stability** | Mean of Std, Std of Std, Std Variation |
| **Relative Variability** | Mean CV, Std CV, CV Variation |
| **Near-Zero Protection** | Mean range, Near-Zero Mean flag |
| **Distributional Stability** | To be evaluated in the next stage |

This produces a richer representation of feature behavior than a single coefficient or statistic.

---

## 6. Important Observation from Current Results

The current results demonstrate that feature stability differs substantially across datasets and features.

Examples include:

- Some features exhibit very low variation in their means and standard deviations, indicating strong statistical stability.
- Features such as `ApplicantIncome` and `CoapplicantIncome` show considerably greater variation across windows.
- In the house-price dataset, features such as `sqft_lot` exhibit greater variability than relatively stable structural variables.
- In the fraud dataset, many anonymized features have means extremely close to zero. Consequently, their CV values are intentionally excluded from interpretation.
- Some fraud features show substantial changes in their window-level standard deviations, suggesting that variance-based stability may provide useful information even when mean-based CV is not meaningful.

These observations support the need for a **multi-statistic reliability framework**.

---

## 7. Research Interpretation

At this stage, the analysis does **not yet assign a final Feature Reliability Score (FRS)**.

Instead, it establishes the individual statistical components required to construct such a score.

The current stage answers:

> **"How stable is a feature's statistical behavior across data windows?"**

The later stages will investigate:

> **"Does this statistical instability correspond to distributional drift, feature-importance instability, and ultimately ML performance degradation?"**

This distinction is important for avoiding premature conclusions about feature reliability.

---

## 8. Next Research Stage

The next stage will extend the stability analysis from summary statistics to **distributional stability**.

For each feature, distributions across windows will be compared using appropriate distribution-based measures.

Planned analysis includes:

- Distribution comparison between windows
- Quantification of distributional shift
- Identification of features whose distributions change despite stable means
- Comparison of statistical stability with distributional stability

The resulting measures will become an additional component of the proposed **Feature Reliability Analyzer**.

---

### Current Research Pipeline

**Raw Data**

↓

**Data Quality Assessment**

↓

**Window Construction**

↓

**Multi-Statistic Feature Stability Analysis**

- Mean Stability
- Standard-Deviation Stability
- CV Stability
- Near-Zero Mean Handling

↓

**Distributional Stability Analysis** ← **NEXT**

↓

**Feature Importance Stability**

↓

**Feature Reliability Score (FRS)**

↓

**ML Performance Evaluation**

↓

**Performance Degradation / Generalization Analysis**

---

### Research Principle

> **Feature reliability should not be determined from a single statistic.**

A feature will be considered more reliable when its behavior remains consistent across multiple dimensions, while instability in one or more dimensions will be investigated further through distributional analysis, importance stability, and model-performance experiments.